# 🌍 CanopyML: Satellite Land Cover Classification & Deforestation Detection

> **A research notebook using ResNet50 Transfer Learning on EuroSAT + Sentinel-2 imagery**

---

## Abstract

This notebook implements a full pipeline for **land cover classification** and **deforestation change detection** using convolutional neural networks. We fine-tune a ResNet50 model pre-trained on ImageNet on the EuroSAT dataset (27,000 satellite image patches across 10 land cover classes). The trained model is then applied to Sentinel-2 satellite imagery from two different years to identify regions where **forest cover was lost**, and the results are validated against Global Forest Watch (GFW) data.

## 📋 Table of Contents
1. Environment Setup & Configuration
2. Dataset Loading & Preprocessing
3. Model Architecture
4. Two-Stage Training
5. Evaluation & Metrics
6. Satellite Inference Pipeline
7. Change Detection
8. Global Forest Watch Validation
9. Results Summary & How to Adapt

---
## Section 1: Environment Setup & Configuration

In [1]:
!pip install rasterio --quiet
print('✅ Dependencies installed')

zsh:1: command not found: pip
✅ Dependencies installed


In [3]:
import os, time, random, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import seaborn as sns

from PIL import Image
import rasterio

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
import torchvision
from torchvision import transforms, datasets, models
from sklearn.metrics import classification_report, confusion_matrix

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED); torch.backends.cudnn.deterministic = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'🖥️  Running on: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'   GPU: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

🖥️  Running on: cpu


In [5]:
# ╔══════════════════════════════════════════════════════════════╗
# ║          CONFIGURATION — Edit these paths as needed          ║
# ╚══════════════════════════════════════════════════════════════╝

EUROSAT_DIR        = '/content/EuroSAT_RGB'
SENTINEL_2018      = '/content/sentinel_2018.tif'
SENTINEL_2024      = '/content/sentinel_2024.tif'
GFW_LOSS_MASK      = '/content/gfw_loss.tif'

CHECKPOINT_PATH    = '/content/eurosat_resnet50.pth'
CONF_MATRIX_PATH   = '/content/confusion_matrix.png'
LANDCOVER_2018_NPY = '/content/landcover_2018.npy'
LANDCOVER_2024_NPY = '/content/landcover_2024.npy'
CHANGE_MAP_PNG     = '/content/change_map.png'
DEFOR_CSV          = '/content/deforestation_coords.csv'
BENCHMARK_PNG      = '/content/benchmark_comparison.png'

BATCH_SIZE           = 32
NUM_CLASSES          = 10
STAGE_A_EPOCHS       = 10
STAGE_A_LR           = 1e-3
STAGE_B_EPOCHS       = 10
STAGE_B_LR           = 1e-4
EARLY_STOP_PATIENCE  = 3

PATCH_SIZE       = 64
PATCH_STRIDE     = 64
INFER_BATCH_SIZE = 128

CLASS_NAMES = [
    'AnnualCrop', 'Forest', 'HerbVeg', 'Highway', 'Industrial',
    'Pasture', 'PermCrop', 'Residential', 'River', 'SeaLake',
]

FOREST_IDX         = 1
NON_FOREST_TARGETS = [0, 2, 4, 5, 6, 7]

LANDCOVER_COLORS = [
    '#f5c242', '#2d8c4e', '#a8d5a2', '#8c8c8c', '#e05c2e',
    '#c8e6c9', '#f9a825', '#ef5350', '#1565c0', '#29b6f6',
]

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

print('✅ Configuration loaded')

✅ Configuration loaded


---
## Section 2: Dataset Loading & Preprocessing

EuroSAT contains 27,000 images across 10 land cover classes. We split 80/10/10 (train/val/test) with a fixed seed. Training images receive data augmentation (flips, rotation, colour jitter) to improve generalisation.

In [6]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

print('✅ Transforms defined')

✅ Transforms defined


In [7]:
full_dataset = datasets.ImageFolder(root=EUROSAT_DIR, transform=test_transform)

TOTAL      = len(full_dataset)
TRAIN_SIZE = int(0.80 * TOTAL)
VAL_SIZE   = int(0.10 * TOTAL)
TEST_SIZE  = TOTAL - TRAIN_SIZE - VAL_SIZE

generator = torch.Generator().manual_seed(SEED)
train_idx, val_idx, test_idx = random_split(
    range(TOTAL), [TRAIN_SIZE, VAL_SIZE, TEST_SIZE], generator=generator
)


class TransformSubset(torch.utils.data.Dataset):
    """Wraps a random_split index object and applies a given transform."""
    def __init__(self, subset, transform):
        self.subset    = subset
        self.transform = transform

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        base          = self.subset.dataset
        old_transform = base.transform
        base.transform = None
        img, label    = base[self.subset.indices[idx]]
        base.transform = old_transform
        return self.transform(img), label


train_dataset = TransformSubset(train_idx, train_transform)
val_dataset   = TransformSubset(val_idx,   test_transform)
test_dataset  = TransformSubset(test_idx,  test_transform)

print(f'📦 Dataset size : {TOTAL:,}')
print(f'   Train        : {len(train_dataset):,}')
print(f'   Val          : {len(val_dataset):,}')
print(f'   Test         : {len(test_dataset):,}')

assert full_dataset.classes == CLASS_NAMES, (
    f'Class order mismatch!\nExpected: {CLASS_NAMES}\nGot: {full_dataset.classes}'
)
print('✅ Class order verified')

FileNotFoundError: [Errno 2] No such file or directory: '/content/EuroSAT_RGB'

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'✅ DataLoaders ready  |  Train: {len(train_loader)} batches  |  Val: {len(val_loader)}  |  Test: {len(test_loader)}')

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
fig.suptitle('EuroSAT — One Sample per Class', fontsize=14, fontweight='bold')

class_samples = {}
for img_path, label in full_dataset.samples:
    if label not in class_samples:
        class_samples[label] = img_path
    if len(class_samples) == NUM_CLASSES:
        break

for ax, (cls_idx, img_path) in zip(axes.flat, sorted(class_samples.items())):
    img = Image.open(img_path).convert('RGB')
    ax.imshow(img)
    ax.set_title(CLASS_NAMES[cls_idx], fontsize=10, fontweight='bold',
                 color=LANDCOVER_COLORS[cls_idx])
    ax.axis('off')

plt.tight_layout()
plt.savefig('/content/sample_patches.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Section 3: Model Architecture

ResNet50 with ImageNet weights. Backbone frozen in Stage A; all layers unfrozen in Stage B.

```
ResNet50 Backbone  →  avgpool  →  flatten  →  [2048-dim]  →  nn.Linear(2048, 10)
```

In [ ]:
def build_model(num_classes=NUM_CLASSES, freeze_backbone=True):
    weights = models.ResNet50_Weights.IMAGENET1K_V1
    model   = models.resnet50(weights=weights)
    if freeze_backbone:
        for param in model.parameters():
            param.requires_grad = False
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model


model = build_model(freeze_backbone=True).to(DEVICE)

total_p     = sum(p.numel() for p in model.parameters())
trainable_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'✅ ResNet50 built')
print(f'   Total parameters     : {total_p:,}')
print(f'   Trainable parameters : {trainable_p:,}  ({100*trainable_p/total_p:.1f}%)')

---
## Section 4: Two-Stage Training

- **Stage A (10 epochs):** Adam lr=0.001, only `model.fc` trained.
- **Stage B (10 epochs + early stopping):** All layers unfrozen, Adam lr=0.0001, patience=3.

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * images.size(0)
    return total_loss / len(loader.dataset)


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    correct = total = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        correct += (model(images).argmax(dim=1) == labels).sum().item()
        total   += labels.size(0)
    return correct / total


def run_training_stage(model, train_loader, val_loader, epochs, lr, device,
                       stage_name, patience=None, checkpoint_path=None):
    criterion    = nn.CrossEntropyLoss()
    optimizer    = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    history      = {'train_loss': [], 'val_acc': []}
    best_val_acc = 0.0
    patience_ctr = 0

    print(f'\n{"="*60}')
    print(f'  {stage_name}  |  lr={lr}  |  max_epochs={epochs}')
    print(f'{"="*60}')
    print(f'{"Epoch":>6} | {"Train Loss":>12} | {"Val Acc":>10} | {"Time":>8}')
    print(f'{"-"*6}-+{("-"*12)}-+{("-"*10)}-+{("-"*8)}')

    for epoch in range(1, epochs + 1):
        t0          = time.time()
        train_loss  = train_epoch(model, train_loader, criterion, optimizer, device)
        val_acc     = evaluate(model, val_loader, device)
        elapsed     = time.time() - t0
        history['train_loss'].append(train_loss)
        history['val_acc'].append(val_acc)

        flag = ''
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            patience_ctr = 0
            flag         = ' ★ best'
            if checkpoint_path:
                torch.save(model.state_dict(), checkpoint_path)
        else:
            patience_ctr += 1

        print(f'{epoch:>6} | {train_loss:>12.4f} | {val_acc:>9.4f} | {elapsed:>6.1f}s{flag}')

        if patience and patience_ctr >= patience:
            print(f'\n⚠️  Early stopping at epoch {epoch}')
            break

    print(f'\n✅ {stage_name} complete. Best Val Acc: {best_val_acc:.4f}')
    return history


print('✅ Training utilities defined')

In [ ]:
history_a = run_training_stage(
    model=model, train_loader=train_loader, val_loader=val_loader,
    epochs=STAGE_A_EPOCHS, lr=STAGE_A_LR, device=DEVICE,
    stage_name='Stage A — Transfer Learning (head only)',
    patience=None, checkpoint_path=None,
)

In [ ]:
# Unfreeze the full backbone for fine-tuning
for param in model.parameters():
    param.requires_grad = True
print(f'Stage B trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

history_b = run_training_stage(
    model=model, train_loader=train_loader, val_loader=val_loader,
    epochs=STAGE_B_EPOCHS, lr=STAGE_B_LR, device=DEVICE,
    stage_name='Stage B — Fine-Tuning (all layers)',
    patience=EARLY_STOP_PATIENCE, checkpoint_path=CHECKPOINT_PATH,
)

In [ ]:
epochs_a        = range(1, len(history_a['train_loss']) + 1)
offset          = len(history_a['train_loss'])
epochs_b        = range(offset + 1, offset + len(history_b['train_loss']) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('CanopyML Training Curves — ResNet50 on EuroSAT', fontsize=13, fontweight='bold')

ax1.plot(epochs_a, history_a['train_loss'], 'o-', color='#4CAF50', label='Stage A loss')
ax1.plot(epochs_b, history_b['train_loss'], 's-', color='#2196F3', label='Stage B loss')
ax1.axvline(x=offset + 0.5, color='grey', linestyle='--', alpha=0.5, label='Fine-tune start')
ax1.set(xlabel='Epoch', ylabel='Training Loss', title='Training Loss')
ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(epochs_a, [v * 100 for v in history_a['val_acc']], 'o-', color='#4CAF50', label='Stage A')
ax2.plot(epochs_b, [v * 100 for v in history_b['val_acc']], 's-', color='#2196F3', label='Stage B')
ax2.axvline(x=offset + 0.5, color='grey', linestyle='--', alpha=0.5)
ax2.axhline(y=95, color='red', linestyle=':', alpha=0.6, label='95% target')
ax2.set(xlabel='Epoch', ylabel='Validation Accuracy (%)', title='Validation Accuracy')
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/content/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Saved training curves')

---
## Section 5: Evaluation & Metrics

Load the best checkpoint and evaluate on the held-out test set.

In [ ]:
model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=DEVICE))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in test_loader:
        preds = model(images.to(DEVICE)).argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)
test_acc   = (all_preds == all_labels).mean()

print(f'🎯 Test Accuracy: {test_acc * 100:.2f}%')
print('\n📊 Classification Report:')
print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES, digits=4))

In [ ]:
cm      = confusion_matrix(all_labels, all_preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    cm_norm, annot=True, fmt='.1f', cmap='YlOrRd',
    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
    linewidths=0.5, cbar_kws={'label': 'Recall (%)'}, ax=ax
)
ax.set(xlabel='Predicted Label', ylabel='True Label',
       title=f'Confusion Matrix — ResNet50 on EuroSAT (Acc={test_acc*100:.2f}%)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(CONF_MATRIX_PATH, dpi=150, bbox_inches='tight')
plt.show()
print(f'💾 Saved confusion matrix → {CONF_MATRIX_PATH}')

---
## Section 6: Satellite Inference Pipeline

1. `patchify_image()` — slides a 64×64 window (stride=64) over the full image.
2. `infer_landcover()` — GPU batch-inference → 2D land cover grid.
3. `visualize_landcover()` — discrete colormap + legend.

> Each 64×64 patch at Sentinel-2 10 m/px resolution covers **640 m × 640 m ≈ 0.41 km²**.

In [ ]:
def patchify_image(image_path, patch_size=PATCH_SIZE, stride=PATCH_STRIDE):
    """
    Extract non-overlapping patches from a satellite image.
    Supports GeoTIFF (via rasterio) and standard formats (via PIL).

    Returns:
        patches    : list of PIL.Image patches
        coords     : list of (row, col) top-left positions
        grid_shape : (n_rows, n_cols)
        img_array  : full image as (H, W, 3) uint8
    """
    if image_path.lower().endswith(('.tif', '.tiff')):
        with rasterio.open(image_path) as src:
            arr = src.read([1, 2, 3]).transpose(1, 2, 0)
        if arr.dtype != np.uint8:
            p2, p98 = np.percentile(arr, 2), np.percentile(arr, 98)
            arr = np.clip((arr - p2) / (p98 - p2 + 1e-6) * 255, 0, 255).astype(np.uint8)
    else:
        arr = np.array(Image.open(image_path).convert('RGB'))

    H, W, _ = arr.shape
    patches, coords = [], []
    n_rows = (H - patch_size) // stride + 1
    n_cols = (W - patch_size) // stride + 1

    for r in range(0, n_rows * stride, stride):
        for c in range(0, n_cols * stride, stride):
            patches.append(Image.fromarray(arr[r:r+patch_size, c:c+patch_size]))
            coords.append((r, c))

    print(f'   Image  : {W}×{H} px  |  Grid : {n_rows}×{n_cols} = {len(patches)} patches')
    return patches, coords, (n_rows, n_cols), arr


def infer_landcover(model, image_path, device=DEVICE,
                    patch_size=PATCH_SIZE, stride=PATCH_STRIDE,
                    infer_batch=INFER_BATCH_SIZE):
    """Run patchify → GPU inference → reconstruct 2D land cover grid."""
    model.eval()
    print(f'\n🛰️  Inferring: {os.path.basename(image_path)}')
    patches, coords, grid_shape, img_array = patchify_image(image_path, patch_size, stride)

    all_preds = []
    with torch.no_grad():
        for i in range(0, len(patches), infer_batch):
            batch = torch.stack([test_transform(p) for p in patches[i:i+infer_batch]]).to(device)
            all_preds.extend(model(batch).argmax(dim=1).cpu().numpy())

    landcover_grid = np.array(all_preds).reshape(grid_shape)
    print(f'   ✅ Done. Grid shape: {landcover_grid.shape}')
    return landcover_grid, img_array


def visualize_landcover(grid, title='Land Cover Map', save_path=None):
    cmap   = mcolors.ListedColormap(LANDCOVER_COLORS)
    norm   = mcolors.BoundaryNorm(np.arange(-0.5, NUM_CLASSES, 1), cmap.N)
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.imshow(grid, cmap=cmap, norm=norm, interpolation='nearest')
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.axis('off')
    patches = [mpatches.Patch(color=LANDCOVER_COLORS[i], label=CLASS_NAMES[i]) for i in range(NUM_CLASSES)]
    ax.legend(handles=patches, bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f'💾 Saved → {save_path}')
    plt.show()


print('✅ Inference pipeline functions defined')

In [ ]:
if os.path.exists(SENTINEL_2018):
    landcover_2018, img_arr_2018 = infer_landcover(model, SENTINEL_2018)
    np.save(LANDCOVER_2018_NPY, landcover_2018)
    visualize_landcover(landcover_2018, 'Land Cover Map — 2018 (Sentinel-2)', '/content/landcover_map_2018.png')
else:
    print(f'⚠️  {SENTINEL_2018} not found — generating demo grid')
    np.random.seed(SEED)
    landcover_2018 = np.random.randint(0, NUM_CLASSES, (50, 50))
    landcover_2018[10:25, 10:30] = FOREST_IDX   # simulate forest region
    img_arr_2018   = np.zeros((50*PATCH_SIZE, 50*PATCH_SIZE, 3), dtype=np.uint8)
    np.save(LANDCOVER_2018_NPY, landcover_2018)
    visualize_landcover(landcover_2018, '[DEMO] Land Cover Map — 2018', '/content/landcover_map_2018.png')

In [ ]:
if os.path.exists(SENTINEL_2024):
    landcover_2024, img_arr_2024 = infer_landcover(model, SENTINEL_2024)
    np.save(LANDCOVER_2024_NPY, landcover_2024)
    visualize_landcover(landcover_2024, 'Land Cover Map — 2024 (Sentinel-2)', '/content/landcover_map_2024.png')
else:
    print(f'⚠️  {SENTINEL_2024} not found — generating demo grid with simulated deforestation')
    landcover_2024 = landcover_2018.copy()
    landcover_2024[12:22, 12:25] = 0   # AnnualCrop (agricultural conversion)
    landcover_2024[15:20, 25:28] = 5   # Pasture (cattle ranching)
    img_arr_2024 = np.zeros_like(img_arr_2018)
    np.save(LANDCOVER_2024_NPY, landcover_2024)
    visualize_landcover(landcover_2024, '[DEMO] Land Cover Map — 2024', '/content/landcover_map_2024.png')

---
## Section 7: Change Detection

Pixel-wise comparison: flag grid cells where class changed **from Forest (1) to non-forest**.

| Destination | Interpretation |
|---|---|
| AnnualCrop (0) | Agricultural conversion |
| HerbVeg (2)    | Degradation / savannification |
| Industrial (4) | Industrial development |
| Pasture (5)    | Cattle ranching |
| PermCrop (6)   | Plantation establishment |
| Residential (7)| Urbanization |

In [ ]:
min_rows = min(landcover_2018.shape[0], landcover_2024.shape[0])
min_cols = min(landcover_2018.shape[1], landcover_2024.shape[1])
grid_2018 = landcover_2018[:min_rows, :min_cols]
grid_2024 = landcover_2024[:min_rows, :min_cols]
print(f'Aligned grid shape: {grid_2018.shape}')

was_forest        = (grid_2018 == FOREST_IDX)
became_nonforest  = np.isin(grid_2024, NON_FOREST_TARGETS)
deforestation_mask = was_forest & became_nonforest

n_deforested = int(deforestation_mask.sum())
area_km2     = n_deforested * (PATCH_SIZE * 10)**2 / 1e6

print(f'\n🌲 Change Detection Results')
print(f'   Forest patches (2018)         : {int(was_forest.sum()):,}')
print(f'   Deforestation events detected : {n_deforested:,}')
print(f'   Estimated deforested area     : {area_km2:.2f} km²')

In [ ]:
cmap  = mcolors.ListedColormap(LANDCOVER_COLORS)
norm  = mcolors.BoundaryNorm(np.arange(-0.5, NUM_CLASSES, 1), cmap.N)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Deforestation Change Detection: 2018 → 2024', fontsize=14, fontweight='bold')

axes[0].imshow(grid_2018, cmap=cmap, norm=norm, interpolation='nearest')
axes[0].set_title('Land Cover 2018', fontsize=11, fontweight='bold'); axes[0].axis('off')

axes[1].imshow(grid_2024, cmap=cmap, norm=norm, interpolation='nearest')
axes[1].set_title('Land Cover 2024', fontsize=11, fontweight='bold'); axes[1].axis('off')

# Overlay: grey base + red deforestation highlight
overlay = np.stack([grid_2024 / (NUM_CLASSES - 1)] * 3, axis=-1).copy()
overlay[deforestation_mask] = [1.0, 0.1, 0.1]
axes[2].imshow(overlay, interpolation='nearest')
axes[2].set_title(f'Deforestation Mask  (n={n_deforested:,} patches)', fontsize=11, fontweight='bold')
axes[2].axis('off')

legend_patches = [mpatches.Patch(color=LANDCOVER_COLORS[i], label=CLASS_NAMES[i]) for i in range(NUM_CLASSES)]
fig.legend(handles=legend_patches, loc='lower center', ncol=5, fontsize=8,
           bbox_to_anchor=(0.5, -0.08))

plt.tight_layout()
plt.savefig(CHANGE_MAP_PNG, dpi=150, bbox_inches='tight')
plt.show()
print(f'💾 Saved → {CHANGE_MAP_PNG}')

In [ ]:
defor_rows, defor_cols = np.where(deforestation_mask)

defor_df = pd.DataFrame({
    'grid_row'  : defor_rows,
    'grid_col'  : defor_cols,
    'pixel_x'   : defor_cols * PATCH_STRIDE,
    'pixel_y'   : defor_rows * PATCH_STRIDE,
    'class_2018': [CLASS_NAMES[grid_2018[r, c]] for r, c in zip(defor_rows, defor_cols)],
    'class_2024': [CLASS_NAMES[grid_2024[r, c]] for r, c in zip(defor_rows, defor_cols)],
    'area_m2'   : PATCH_SIZE * PATCH_SIZE * 100,  # 10 m/px → (px²) × (10m)² = m²
})

defor_df.to_csv(DEFOR_CSV, index=False)
print(f'💾 Saved {len(defor_df):,} deforestation events → {DEFOR_CSV}')
print(defor_df.head(10).to_string(index=False))

---
## Section 8: Global Forest Watch Validation

Cross-reference our detections against GFW Hansen tree cover loss data.

| Metric | Formula |
|---|---|
| Precision | TP / (TP + FP) |
| Recall    | TP / (TP + FN) |
| F1-Score  | 2PR / (P+R) |
| Agreement | TP / (TP+FP+FN) = IoU |

In [ ]:
def load_gfw_mask(gfw_path, target_shape):
    """
    Load GFW binary mask and resize to match the patch grid.
    Uses nearest-neighbour resampling to preserve binary values.
    """
    with rasterio.open(gfw_path) as src:
        gfw_raw = src.read(1)
    gfw_img     = Image.fromarray(gfw_raw.astype(np.uint8))
    gfw_resized = gfw_img.resize((target_shape[1], target_shape[0]), Image.NEAREST)
    return np.array(gfw_resized).astype(bool)


if os.path.exists(GFW_LOSS_MASK):
    gfw_grid = load_gfw_mask(GFW_LOSS_MASK, deforestation_mask.shape)
    print(f'✅ GFW mask loaded  |  shape: {gfw_grid.shape}  |  loss pixels: {gfw_grid.sum():,}')
else:
    print(f'⚠️  {GFW_LOSS_MASK} not found — generating synthetic GFW mask for demo')
    np.random.seed(SEED + 2)
    gfw_grid = np.zeros(deforestation_mask.shape, dtype=bool)
    defor_idx = np.argwhere(deforestation_mask)
    if len(defor_idx) > 0:
        n_tp  = int(0.70 * len(defor_idx))
        chosen = defor_idx[np.random.choice(len(defor_idx), n_tp, replace=False)]
        gfw_grid[chosen[:, 0], chosen[:, 1]] = True
    gfw_grid[5:8, 5:8] = True   # some GFW-only events
    print(f'   Synthetic GFW loss pixels: {gfw_grid.sum():,}')

In [ ]:
TP = int((deforestation_mask &  gfw_grid).sum())
FP = int((deforestation_mask & ~gfw_grid).sum())
FN = int((~deforestation_mask & gfw_grid).sum())
TN = int((~deforestation_mask & ~gfw_grid).sum())

precision = TP / (TP + FP + 1e-9)
recall    = TP / (TP + FN + 1e-9)
f1        = 2 * precision * recall / (precision + recall + 1e-9)
iou       = TP / (TP + FP + FN + 1e-9)

print('\n📊 GFW Validation Summary')
print('═' * 45)
print(f'  True Positives  (TP) : {TP:>6,}')
print(f'  False Positives (FP) : {FP:>6,}   ← our detections not in GFW')
print(f'  False Negatives (FN) : {FN:>6,}   ← GFW events we missed')
print(f'  True Negatives  (TN) : {TN:>6,}')
print('─' * 45)
print(f'  Precision   : {precision:.4f}  ({precision*100:.1f}%)')
print(f'  Recall      : {recall:.4f}  ({recall*100:.1f}%)')
print(f'  F1-Score    : {f1:.4f}  ({f1*100:.1f}%)')
print(f'  Agreement   : {iou:.4f}  ({iou*100:.1f}%)  [IoU]')
print('═' * 45)

pd.DataFrame({
    'Metric': ['TP', 'FP', 'FN', 'TN', 'Precision', 'Recall', 'F1-Score', 'IoU'],
    'Value' : [TP, FP, FN, TN, f'{precision:.4f}', f'{recall:.4f}', f'{f1:.4f}', f'{iou:.4f}']
}).to_csv('/content/gfw_validation_metrics.csv', index=False)
print('\n💾 Metrics saved → /content/gfw_validation_metrics.csv')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Global Forest Watch Benchmark Comparison', fontsize=14, fontweight='bold')

axes[0].imshow(deforestation_mask, cmap='Reds', interpolation='nearest')
axes[0].set_title('(a) Model Detections', fontsize=11, fontweight='bold'); axes[0].axis('off')

axes[1].imshow(gfw_grid, cmap='YlOrBr', interpolation='nearest')
axes[1].set_title('(b) GFW Reference', fontsize=11, fontweight='bold'); axes[1].axis('off')

agree_map = np.zeros((*deforestation_mask.shape, 3), dtype=np.float32)
agree_map[ deforestation_mask &  gfw_grid]  = [0.10, 0.80, 0.10]  # TP: green
agree_map[ deforestation_mask & ~gfw_grid]  = [0.90, 0.10, 0.10]  # FP: red
agree_map[~deforestation_mask &  gfw_grid]  = [0.90, 0.80, 0.00]  # FN: yellow
agree_map[~deforestation_mask & ~gfw_grid]  = [0.95, 0.95, 0.95]  # TN: grey

axes[2].imshow(agree_map, interpolation='nearest')
axes[2].set_title('(c) Agreement / Disagreement', fontsize=11, fontweight='bold')
axes[2].axis('off')

legend_c = [
    mpatches.Patch(color=(0.10, 0.80, 0.10), label=f'TP — Agreement  (n={TP:,})'),
    mpatches.Patch(color=(0.90, 0.10, 0.10), label=f'FP — Our model only (n={FP:,})'),
    mpatches.Patch(color=(0.90, 0.80, 0.00), label=f'FN — GFW only   (n={FN:,})'),
    mpatches.Patch(color=(0.95, 0.95, 0.95), label='TN — No loss'),
]
axes[2].legend(handles=legend_c, loc='lower right', fontsize=8, framealpha=0.9)

metric_text = (f'Precision: {precision*100:.1f}%\nRecall: {recall*100:.1f}%\n'
               f'F1: {f1*100:.1f}%\nAgreement: {iou*100:.1f}%')
axes[2].text(0.02, 0.98, metric_text, transform=axes[2].transAxes, fontsize=9,
             va='top', bbox=dict(boxstyle='round,pad=0.4', fc='white', alpha=0.85))

plt.tight_layout()
plt.savefig(BENCHMARK_PNG, dpi=150, bbox_inches='tight')
plt.show()
print(f'💾 Saved → {BENCHMARK_PNG}')

---
## Section 9: Results Summary & Discussion

### Strengths
- ImageNet transfer learning provides strong low-level feature extractors that generalise well to satellite imagery.
- Two-stage fine-tuning prevents catastrophic forgetting.
- Sliding-window patchify scales to arbitrarily large images.

### Limitations
- EuroSAT was acquired over European landscapes — tropical generalisation may need domain adaptation.
- 64×64 patch classification misses long-range context (consider ViT or full semantic segmentation).
- GFW data is annual cumulative and may not align with exact Sentinel-2 acquisition dates.

---

## 🔧 How to Adapt for a Different Study Area

| Step | Action |
|---|---|
| New area | Update `SENTINEL_2018` / `SENTINEL_2024` paths; acquire imagery via Copernicus or GEE |
| New time period | Change imagery and update legend strings |
| Different resolution | Adjust `PATCH_SIZE` / `PATCH_STRIDE` and recalculate `area_km2` |
| Different target transition | Update `FOREST_IDX` and `NON_FOREST_TARGETS` |
| Different dataset | Swap `EUROSAT_DIR` for any `ImageFolder`-compatible dataset; update `CLASS_NAMES` and `NUM_CLASSES` |
| Different GFW layer | Update `GFW_LOSS_MASK`; `load_gfw_mask()` handles any binary GeoTIFF |

In [ ]:
print('\n' + '='*60)
print('  CanopyML — Output File Summary')
print('='*60)

outputs = [
    ('/content/sample_patches.png',         'EuroSAT sample patch grid'),
    ('/content/training_curves.png',        'Stage A + B training curves'),
    (CONF_MATRIX_PATH,                      'Confusion matrix heatmap'),
    ('/content/landcover_map_2018.png',     'Land cover map 2018'),
    ('/content/landcover_map_2024.png',     'Land cover map 2024'),
    (LANDCOVER_2018_NPY,                    'Land cover grid 2018 [.npy]'),
    (LANDCOVER_2024_NPY,                    'Land cover grid 2024 [.npy]'),
    (CHANGE_MAP_PNG,                        'Deforestation change map'),
    (DEFOR_CSV,                             'Deforestation event coords [.csv]'),
    (BENCHMARK_PNG,                         'GFW benchmark comparison figure'),
    ('/content/gfw_validation_metrics.csv', 'GFW validation metrics [.csv]'),
    (CHECKPOINT_PATH,                       'Best model checkpoint [.pth]'),
]

for path, desc in outputs:
    status = '✅' if os.path.exists(path) else '⚠️ '
    print(f'  {status}  {os.path.basename(path):<42s} {desc}')

print('\n✅ CanopyML pipeline complete!')